# Influencer or Observer — Final Solution (0.843 accuracy)

End-to-end pipeline that produces `submission.csv`:

1. **Behavioural features** (~50) from user & tweet metadata (follower/friend ratios, engagement, account age, profile completeness, bio keywords…)
2. **Text representations** of `tweet [SEP] user description`:
   - `cardiffnlp/twitter-roberta-base-sentiment-latest` [CLS] embeddings → PCA (50 dims)
   - TF-IDF (1–2 grams, 10k features) → TruncatedSVD (100 dims)
3. **K-Means** (12 clusters) on the RoBERTa embeddings, one-hot encoded
4. **Ensemble** of LightGBM + XGBoost + CatBoost, weights optimised with `scipy.optimize`, decision threshold tuned on a validation split
5. **Safe pseudo-labelling**: test predictions with p > 0.95 or p < 0.05 are added to the training set, then the ensemble is retrained

Data must be placed in `../data/` (see `data/README.md`). RoBERTa embeddings are cached in `../data/*.npy` after the first run; a GPU is strongly recommended for that step.


In [ ]:
# IMPORTS

import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from scipy.optimize import minimize
from transformers import AutoTokenizer, AutoModel
import torch
from tqdm import tqdm
import os
import gc
import re

In [ ]:
# DATA PATHS

train_data_path = '../data/train.jsonl'
test_data_path = '../data/kaggle_test.jsonl'

# Cache for RoBERTa embeddings (computed once, then reloaded) and output file
train_emb_path = '../data/train_embeddings.npy'
test_emb_path = '../data/test_embeddings.npy'
submission_path = '../submission.csv'

In [ ]:
# SET SEED AND DEVICE

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Device selected: {DEVICE}")

In [ ]:
# UTILITY FUNCTIONS

def save_embeddings(path, arr):
    """ Save embeddings to avoid recomputation. """
    np.save(path, arr)
    print(f"Saved: {path}")

def load_embeddings(path):
    """ Load precomputed embeddings. """
    arr = np.load(path)
    print(f"Loaded: {path}")
    return arr

def safe_get(d, key, default=0):
    """ Fetch value from dict safely, if not found return default. """
    if isinstance(d, dict):
        return d.get(key, default)
    return default

In [ ]:
# FEATURE ENGINEERING

def extract_influence_features(df):
    """ Extract features related to user influence and tweet metadata. """
    
    # Basic statistics
    followers = df['user'].apply(lambda x: safe_get(x, 'followers_count', 0))
    friends = df['user'].apply(lambda x: max(safe_get(x, 'friends_count', 1), 1))
    statuses = df['user'].apply(lambda x: max(safe_get(x, 'statuses_count', 1), 1))
    listed = df['user'].apply(lambda x: safe_get(x, 'listed_count', 0))
    favourites = df['user'].apply(lambda x: safe_get(x, 'favourites_count', 0))
    verified = df['user'].apply(lambda x: int(safe_get(x, 'verified', False)))
    default_profile = df['user'].apply(lambda x: int(safe_get(x, 'default_profile', True)))
    default_profile_image = df['user'].apply(lambda x: int(safe_get(x, 'default_profile_image', True)))
    account_age = df['user'].apply(get_account_age)
    retweet_count = df['retweet_count'].fillna(0)
    favorite_count = df['favorite_count'].fillna(0)
    reply_count = df['reply_count'].fillna(0)
    is_reply = df['in_reply_to_status_id'].notna().astype(int)
    is_quote = df['is_quote_status'].fillna(False).astype(int)
    
    # Text metadata
    texts = df.apply(lambda x: get_full_text(x), axis=1)
    text_lengths = texts.apply(len)
    word_counts = texts.apply(lambda x: len(str(x).split()))
    hashtag_count = df['entities'].apply(lambda x: len(x.get('hashtags', [])) if isinstance(x, dict) else 0)
    mention_count = df['entities'].apply(lambda x: len(x.get('user_mentions', [])) if isinstance(x, dict) else 0)
    url_count = df['entities'].apply(lambda x: len(x.get('urls', [])) if isinstance(x, dict) else 0)
    has_media = df.apply(lambda x: 1 if (isinstance(x.get('entities'), dict) and x['entities'].get('media')) or 
                                         (isinstance(x.get('extended_entities'), dict) and x['extended_entities'].get('media')) 
                                    else 0, axis=1)
    has_description = df['user'].apply(lambda x: int(bool(safe_get(x, 'description', ''))))
    description_length = df['user'].apply(lambda x: len(str(safe_get(x, 'description', ''))))
    has_url = df['user'].apply(lambda x: int(bool(safe_get(x, 'url', ''))))
    has_location = df['user'].apply(lambda x: int(bool(safe_get(x, 'location', ''))))
    has_banner = df['user'].apply(lambda x: int(bool(safe_get(x, 'profile_banner_url', ''))))
    source_lower = df['source'].fillna('').astype(str).str.lower()
    is_web = source_lower.str.contains('web').astype(int)
    is_iphone = source_lower.str.contains('iphone').astype(int)
    is_android = source_lower.str.contains('android').astype(int)

    # Detection of keywords in user description for business/social/official indicators
    def check_keywords(text, keywords):
        text = str(text).lower()
        return int(any(k in text for k in keywords))

    business_keywords = ['booking', 'contact', 'business', 'enquiries', 'email', 'gmail', 'manager', 'pr', 'collab', 'ambassador', 'promo', 'discount', 'dm for', 'managed by']
    social_keywords = ['instagram', 'ig:', 'insta:', 'youtube', 'tiktok', 'snapchat', 'twitch', 'sub', 'onlyfans', 'link in bio', 'subscribe', 'follow me']
    official_keywords = ['official', 'account', 'real', 'verified', 'original', 'author', 'artist', 'founder', 'ceo', 'journalist', 'public figure', 'actor', 'musician']

    has_business_keyword = df['user'].apply(lambda x: check_keywords(safe_get(x, 'description', ''), business_keywords))
    has_social_keyword = df['user'].apply(lambda x: check_keywords(safe_get(x, 'description', ''), social_keywords))
    has_official_keyword = df['user'].apply(lambda x: check_keywords(safe_get(x, 'description', ''), official_keywords))

    # Name analysis: length and digit count
    screen_name_len = df['user'].apply(lambda x: len(safe_get(x, 'screen_name', '')))
    digits_in_name = df['user'].apply(lambda x: sum(c.isdigit() for c in safe_get(x, 'screen_name', '')))
    
    # Construct feature DataFrame
    features = pd.DataFrame({
        'followers_count': followers,
        'friends_count': friends,
        'log_followers': np.log1p(followers),
        'log_friends': np.log1p(friends),
        'follower_friend_ratio': followers / friends,
        'log_follower_friend_ratio': np.log1p(followers / friends),
        'follower_minus_friend': followers - friends,
        'listed_count': listed,
        'log_listed': np.log1p(listed),
        'listed_per_follower': listed / np.maximum(followers, 1),
        'retweet_count': retweet_count,
        'favorite_count': favorite_count,
        'reply_count': reply_count,
        'total_engagement': retweet_count + favorite_count + reply_count,
        'engagement_per_follower': (retweet_count + favorite_count) / np.maximum(followers, 1),
        'statuses_count': statuses,
        'log_statuses': np.log1p(statuses),
        'account_age_years': account_age,
        'tweets_per_day': statuses / np.maximum(account_age * 365, 1),
        'favourites_count': favourites,
        'verified': verified,
        'default_profile': default_profile,
        'default_profile_image': default_profile_image,
        'has_description': has_description,
        'description_length': description_length,
        'has_url': has_url,
        'has_location': has_location,
        'has_banner': has_banner,
        'profile_completeness': has_description + has_url + has_location + has_banner,
        'is_reply': is_reply,
        'is_quote': is_quote,
        'text_length': text_lengths,
        'word_count': word_counts,
        'avg_word_length': text_lengths / np.maximum(word_counts, 1),
        'hashtag_count': hashtag_count,
        'mention_count': mention_count,
        'url_count': url_count,
        'has_media': has_media,
        'source_is_web': is_web,
        'source_is_iphone': is_iphone,
        'source_is_android': is_android,
        
        'has_business_keyword': has_business_keyword,
        'has_social_keyword': has_social_keyword,
        'has_official_keyword': has_official_keyword,
        'screen_name_len': screen_name_len,
        'digits_in_name': digits_in_name,
        'ratio_digits_name': digits_in_name / np.maximum(screen_name_len, 1),
        'is_old_account_low_followers': ((account_age > 5) & (followers < 100)).astype(int),
        'influence_score': (np.log1p(followers) * np.log1p(listed)) / np.maximum(np.log1p(friends), 1),
    })
    
    features = features.fillna(0).replace([np.inf, -np.inf], 0).astype(np.float32)
    print(f"{features.shape[1]} features")
    return features

def get_full_text(row):
    """ Get the full text of a tweet, considering extended tweets."""
    text = row.get('text', '')
    if not text:
        text = row.get('full_text', '')
    if not text and isinstance(row.get('extended_tweet'), dict):
        text = row.get('extended_tweet', {}).get('full_text', '')
    return str(text) if text else ''

def get_account_age(user):
    """ Get account age in years from created_at field."""
    try:
        created_at = safe_get(user, 'created_at', '')
        if created_at:
            year = int(str(created_at).split()[-1])
            return 2025 - year
    except:
        pass
    return 1

def create_combined_text(df):
    """ Combine tweet text and user description for text processing. """
    combined = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="   Processing"):
        tweet = get_full_text(row)
        user = row.get('user', {})
        desc = user.get('description', '') if isinstance(user, dict) else ''
        combined.append(f"{tweet} [SEP] {desc}" if desc else tweet)
    return combined

In [ ]:
# TEXT PROCESSING

def encode_texts_roberta(texts, batch_size=128, max_length=128):
    """ Encode texts using RoBERTa model to get embeddings. """
    
    tokenizer = AutoTokenizer.from_pretrained('cardiffnlp/twitter-roberta-base-sentiment-latest')
    model = AutoModel.from_pretrained('cardiffnlp/twitter-roberta-base-sentiment-latest')
    model.to(DEVICE)
    model.eval()
    
    embeddings = []
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc="Batches"):
            batch = texts[i:i+batch_size]
            encoded = tokenizer(batch, padding=True, truncation=True, max_length=max_length, return_tensors='pt')
            input_ids = encoded['input_ids'].to(DEVICE)
            attention_mask = encoded['attention_mask'].to(DEVICE)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            embeddings.append(outputs.last_hidden_state[:, 0, :].cpu().numpy().astype(np.float32))

    embeddings = np.vstack(embeddings)   
    return embeddings

def process_tfidf(train_texts, test_texts, n_components=100):
    """ Compute TF-IDF features and reduce dimensionality with SVD. """
    
    def clean(t):
        """ Clean text by lowercasing and removing URLs using regular expressions. """
        t = str(t).lower()
        t = re.sub(r'http\S+|www\S+|https\S+', '', t, flags=re.MULTILINE)
        return t

    train_clean = [clean(t) for t in train_texts]
    test_clean = [clean(t) for t in test_texts]
    
    tfidf = TfidfVectorizer(max_features=10000, stop_words='english', ngram_range=(1, 2))
    train_tfidf = tfidf.fit_transform(train_clean)
    test_tfidf = tfidf.transform(test_clean)
    
    svd = TruncatedSVD(n_components=n_components, random_state=42)
    train_svd = svd.fit_transform(train_tfidf).astype(np.float32)
    test_svd = svd.transform(test_tfidf).astype(np.float32)
    
    print(f"TF-IDF SVD Explained Variance: {sum(svd.explained_variance_ratio_):.2f}")
    return train_svd, test_svd

In [ ]:
# MODEL TRAINING AND ENSEMBLING

def train_stack_model(X_train, y_train, X_val=None, y_val=None, full_train=False):
    """ Train ensemble of LightGBM, XGBoost, and CatBoost models. """
    models = {}
    
    # 1. LightGBM
    print("Training LightGBM")
    lgb_params = {
        'objective': 'binary', 
        'metric': 'binary_logloss', 
        'learning_rate': 0.015,
        'num_leaves': 25,
        'max_depth': 6, 
        'feature_fraction': 0.6, 
        'bagging_fraction': 0.7, 
        'lambda_l1': 0.1,
        'lambda_l2': 0.2,
        'feature_pre_filter': False,
        'verbose': -1, 'seed': 42
    }
    
    if full_train:
        dtrain = lgb.Dataset(X_train, label=y_train)
        models['lgb'] = lgb.train(lgb_params, dtrain, num_boost_round=1500)
    else:
        dtrain = lgb.Dataset(X_train, label=y_train)
        dval = lgb.Dataset(X_val, label=y_val, reference=dtrain)
        models['lgb'] = lgb.train(lgb_params, dtrain, num_boost_round=1500, 
                                valid_sets=[dval], callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])

    # 2. XGBoost
    print("Training XGBoost")
    xgb_params = {
        'objective': 'binary:logistic', 
        'eval_metric': 'logloss',
        'learning_rate': 0.02, 
        'max_depth': 7, 
        'subsample': 0.75, 
        'colsample_bytree': 0.75, 
        'reg_alpha': 0.1, 
        'reg_lambda': 1.0,
        'n_estimators': 1200 if full_train else 1500, 
        'tree_method': 'hist', 
        'random_state': 42, 
        'n_jobs': -1
    }
    
    model_xgb = xgb.XGBClassifier(**xgb_params)
    
    if full_train:
        model_xgb.fit(X_train, y_train, verbose=False)
    else:
        xgb_params['early_stopping_rounds'] = 100
        model_xgb = xgb.XGBClassifier(**xgb_params)
        model_xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    models['xgb'] = model_xgb

    # 3. CatBoost
    print("Training CatBoost")
    cb_params = {
        'loss_function': 'Logloss',
        'eval_metric': 'Accuracy',
        'learning_rate': 0.025, 
        'depth': 7, 
        'l2_leaf_reg': 5,
        'iterations': 1500,
        'random_seed': 42, 
        'verbose': 0, 
        'allow_writing_files': False
    }
    model_cb = CatBoostClassifier(**cb_params)
    if full_train:
        model_cb.fit(X_train, y_train)
    else:
        model_cb.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=100)
    models['cb'] = model_cb
    
    return models


def predict_ensemble(models, X, weights=None):
    """ Finds the ensemble predictions with given weights. """
    p_lgb = models['lgb'].predict(X)
    p_xgb = models['xgb'].predict_proba(X)[:, 1]
    p_cb = models['cb'].predict_proba(X)[:, 1]
    
    if weights is None:
        # Default equal weights
        return (0.33 * p_lgb) + (0.33 * p_xgb) + (0.33 * p_cb)
    else:
        # Optimized weights
        return (weights[0] * p_lgb) + (weights[1] * p_xgb) + (weights[2] * p_cb)


def find_optimal_weights(models, X_val, y_val):
    """ Optimize ensemble weights to maximize accuracy on validation set. """
    
    # Retrieve individual model predictions
    p_lgb = models['lgb'].predict(X_val)
    p_xgb = models['xgb'].predict_proba(X_val)[:, 1]
    p_cb = models['cb'].predict_proba(X_val)[:, 1]
    
    preds_matrix = np.column_stack([p_lgb, p_xgb, p_cb])
    
    # Function to minimize (Negative Accuracy)
    def loss_func(weights):
        # Normalization to ensure sum = 1
        weights = np.array(weights)
        weights /= np.sum(weights)
        final_prob = np.dot(preds_matrix, weights)
        final_pred = (final_prob > 0.5).astype(int)
        return -accuracy_score(y_val, final_pred)

    # Initialization and constraints
    init_weights = [0.33, 0.33, 0.33]
    bounds = [(0.0, 1.0)] * 3
    
    # Optimization
    res = minimize(loss_func, init_weights, bounds=bounds, method='SLSQP')
    
    # Extraction of optimal weights
    best_weights = res.x / np.sum(res.x)
    
    print(f"OPTIMAL WEIGHTS FOUND:")
    print(f"LightGBM : {best_weights[0]:.4f}")
    print(f"XGBoost  : {best_weights[1]:.4f}")
    print(f"CatBoost : {best_weights[2]:.4f}")
    
    return best_weights

In [ ]:
# MAIN PIPELINE

print("Loading data")
train_raw = pd.read_json(train_data_path, lines=True)
test_raw = pd.read_json(test_data_path, lines=True)
    
print("Extracting features")
X_train_feat = extract_influence_features(train_raw)
X_test_feat = extract_influence_features(test_raw)

print("Text processing")
train_texts = create_combined_text(train_raw)
test_texts = create_combined_text(test_raw)

# Load embeddings if they exist, else compute and save
if os.path.exists(train_emb_path):
    train_emb = load_embeddings(train_emb_path)
else:
    train_emb = encode_texts_roberta(train_texts)
    save_embeddings(train_emb_path, train_emb)

if os.path.exists(test_emb_path):
    test_emb = load_embeddings(test_emb_path)
else:
    test_emb = encode_texts_roberta(test_texts)
    save_embeddings(test_emb_path, test_emb)

# PCA on RoBERTa embeddings
print("Applying PCA (50 dims)")
pca = PCA(n_components=50, random_state=42)
train_emb_pca = pca.fit_transform(train_emb).astype(np.float32)
test_emb_pca = pca.transform(test_emb).astype(np.float32)

    
# K-MEANS CLUSTERING 
print("Computing K-Means Clusters")
kmeans = KMeans(n_clusters=12, random_state=42, n_init=10)
all_emb_pca = np.vstack([train_emb_pca, test_emb_pca])
kmeans.fit(all_emb_pca)
train_clusters = kmeans.predict(train_emb_pca)
test_clusters = kmeans.predict(test_emb_pca)
train_clust_df = pd.get_dummies(train_clusters, prefix='cluster').astype(np.float32)
test_clust_df = pd.get_dummies(test_clusters, prefix='cluster').astype(np.float32)

# TF-IDF SVD
train_tfidf_svd, test_tfidf_svd = process_tfidf(train_texts, test_texts, n_components=100)

# Assembling
print("Assembling final datasets")
cols_pca = [f'emb_{i}' for i in range(50)]
cols_tfidf = [f'tfidf_{i}' for i in range(100)]

train_pca_df = pd.DataFrame(train_emb_pca, columns=cols_pca)
test_pca_df = pd.DataFrame(test_emb_pca, columns=cols_pca)

train_tfidf_df = pd.DataFrame(train_tfidf_svd, columns=cols_tfidf)
test_tfidf_df = pd.DataFrame(test_tfidf_svd, columns=cols_tfidf)

X_train_full = pd.concat([X_train_feat.reset_index(drop=True), train_pca_df, train_tfidf_df, train_clust_df.reset_index(drop=True)], axis=1)
X_test = pd.concat([X_test_feat.reset_index(drop=True), test_pca_df, test_tfidf_df, test_clust_df.reset_index(drop=True)], axis=1)
y_train_full = train_raw['label'].values

print(f"Total input features: {X_train_full.shape[1]}")


# PHASE 1 : VALIDATION AND OPTIMIZATION
print("PHASE 1 : Validation Split & Optimization")
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.15, random_state=42, stratify=y_train_full
)
models_val = train_stack_model(X_train, y_train, X_val, y_val, full_train=False)

#  Weights Optimization
optimal_weights = find_optimal_weights(models_val, X_val, y_val)

val_preds = predict_ensemble(models_val, X_val, optimal_weights)

# Optimal Threshold Search
best_acc = 0
best_thresh = 0.5
for thresh in np.arange(0.40, 0.60, 0.005):
    acc = accuracy_score(y_val, (val_preds > thresh).astype(int))
    if acc > best_acc:
        best_acc = acc
        best_thresh = thresh
print(f"Best Threshold: {best_thresh:.3f} | Accuracy: {best_acc:.4f}")



# PHASE 2: INITIAL FULL TRAIN
print("\n" + "="*70)
print("PHASE 2: Full Train")
models_full = train_stack_model(X_train_full, y_train_full, full_train=True)
test_probs = predict_ensemble(models_full, X_test, optimal_weights)


# PHASE 3: SAFE PSEUDO-LABELING
# Select only very high-confidence predictions from the ensemble
print("\n" + "="*70)
print("PHASE 3: SAFE PSEUDO-LABELING (0.95 / 0.05)")

high_conf_indices = np.where((test_probs > 0.95) | (test_probs < 0.05))[0]
pseudo_labels = (test_probs[high_conf_indices] > 0.5).astype(int)

X_pseudo = X_test.iloc[high_conf_indices]
y_pseudo = pseudo_labels

X_train_augmented = pd.concat([X_train_full, X_pseudo], axis=0)
y_train_augmented = np.concatenate([y_train_full, y_pseudo])

print("\n Retraining Trinity on Augmented Data...")
# Retrain the full ensemble (LightGBM/XGBoost/CatBoost) on the dataset augmented with pseudo-labelled examples
models_final = train_stack_model(X_train_augmented, y_train_augmented, full_train=True)
final_probs = predict_ensemble(models_final, X_test, optimal_weights)

test_final = (final_probs > best_thresh).astype(int)

In [ ]:
# SUBMISSION
print("\n" + "="*70)
print("GENERATING SUBMISSION...")
possible_ids = ['challenge_id']
id_col = next((col for col in possible_ids if col in test_raw.columns), None)

if id_col:
    print(f"ID column found: {id_col}")
    ids = test_raw[id_col]
else:
    print("No ID column found, using Index.")
    ids = test_raw.index
    
submission = pd.DataFrame({'ID': ids, 'Prediction': test_final})
submission.to_csv(submission_path, index=False)

print(f"\nSUBMISSION SAVED: {submission_path}")
print(f"Influencers: {(test_final==1).sum():,} ({test_final.mean()*100:.1f}%)")